# Optical Flow

> **Advanced · Video**


## Why this matters

Optical flow estimates apparent motion rather than object identity. It is powerful for motion analysis but rests on assumptions students should learn to question.

**Where it appears:** Stabilization cues, motion visualization, activity measures, and temporal scene analysis.


## Learning Objectives

- Compute sparse optical flow (Lucas-Kanade) for tracked keypoints
- Compute dense optical flow (Farneback) for whole-frame motion fields
- Visualize and interpret flow vectors and magnitude/direction


## Prerequisites

13 Video Processing and Background Motion; 14 Object Tracking

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

Lucas–Kanade flow, Farnebäck flow, feature points, flow magnitude and angle

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Optical Flow

Optical flow estimates per-pixel (or per-keypoint) motion between
consecutive frames, based on the brightness constancy assumption
(a point's intensity is approximately constant as it moves a small amount).
**Sparse** flow (Lucas-Kanade, `cv2.calcOpticalFlowPyrLK`) tracks a
specific set of keypoints efficiently. **Dense** flow (Farneback,
`cv2.calcOpticalFlowFarneback`) estimates motion for every pixel, more
expensive but useful for whole-scene motion analysis (e.g. detecting
overall direction of motion, video stabilization).


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


### 1. Sparse optical flow with Lucas-Kanade

Seed keypoints with `cv2.goodFeaturesToTrack` on frame 0, then propagate them frame-to-frame -- the moving synthetic blob edge provides genuine trackable corners.


In [ ]:
import cv2
import numpy as np
from cv_utils import read_real_video_frames, load_real_image, get_real_data, show_grid

frames = read_real_video_frames("traffic.mp4", max_frames=30)
gray_frames = [cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in frames]

lk_params = dict(
    winSize=(15, 15),
    maxLevel=2,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03),
)

p0 = cv2.goodFeaturesToTrack(
    gray_frames[0], maxCorners=30, qualityLevel=0.2, minDistance=5
)
print(f"Seeded {0 if p0 is None else len(p0)} keypoints to track")

trajectory_canvas = frames[0].copy()
p_prev = p0
for i in range(1, len(gray_frames)):
    if p_prev is None or len(p_prev) == 0:
        break
    p_next, status, err = cv2.calcOpticalFlowPyrLK(
        gray_frames[i - 1], gray_frames[i], p_prev, None, **lk_params
    )
    good_prev = p_prev[status == 1]
    good_next = p_next[status == 1]
    for (x0, y0), (x1, y1) in zip(good_prev.reshape(-1, 2), good_next.reshape(-1, 2)):
        cv2.line(
            trajectory_canvas, (int(x0), int(y0)), (int(x1), int(y1)), (0, 255, 255), 1
        )
    p_prev = good_next.reshape(-1, 1, 2)

show_grid([("Lucas-Kanade tracked point trajectories", trajectory_canvas)], cols=1)

### 2. Dense optical flow with Farneback

Compute a full per-pixel flow field between two frames and convert it to an HSV visualization: hue encodes direction, value encodes magnitude.


In [ ]:
def flow_to_bgr(flow: np.ndarray) -> np.ndarray:
    """Standard optical-flow-to-color visualization: direction -> hue, magnitude -> brightness."""
    magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros((*flow.shape[:2], 3), dtype=np.uint8)
    hsv[..., 0] = angle * 180 / np.pi / 2
    hsv[..., 1] = 255
    hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)


flow = cv2.calcOpticalFlowFarneback(
    gray_frames[10],
    gray_frames[13],
    None,
    pyr_scale=0.5,
    levels=3,
    winsize=15,
    iterations=3,
    poly_n=5,
    poly_sigma=1.2,
    flags=0,
)
flow_vis = flow_to_bgr(flow)
show_grid(
    [("frame 10", frames[10]), ("frame 13", frames[13]), ("dense flow field", flow_vis)]
)

### 3. Extracting a motion summary from dense flow

Reduce the dense flow field to a single 'dominant motion direction and speed' summary -- useful for coarse scene-level motion analysis (e.g. camera pan detection).


In [ ]:
def summarize_flow(flow: np.ndarray) -> dict:
    magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    significant = magnitude > np.percentile(
        magnitude, 90
    )  # focus on the most-moving pixels
    if not significant.any():
        return {"mean_speed": 0.0, "dominant_angle_deg": None}
    return {
        "mean_speed": float(magnitude[significant].mean()),
        "dominant_angle_deg": float(np.degrees(angle[significant].mean())),
    }


print(summarize_flow(flow))

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Optical Flow: Dense Optical Flow Color Wheel Visualization

Dense optical flow produces a displacement vector $(u, v)$ for every pixel. To visualize these vectors intuitively, we map their direction to Hue and their magnitude to Value in the HSV color space, creating a motion color-wheel.


In [ ]:
# Generate a short motion video
frames = read_real_video_frames("traffic.mp4", max_frames=5)
prev = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
curr = cv2.cvtColor(frames[1], cv2.COLOR_BGR2GRAY)

# Calculate Farneback dense optical flow
flow = cv2.calcOpticalFlowFarneback(prev, curr, None, 0.5, 3, 15, 3, 5, 1.2, 0)

# Create flow HSV canvas
h, w = prev.shape[:2]
hsv = np.zeros((h, w, 3), dtype=np.uint8)
hsv[..., 1] = 255  # Keep saturation full

# Compute magnitude and angle
magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1], angleInDegrees=True)

# Map direction to Hue (0-179 range) and magnitude to Value
hsv[..., 0] = angle / 2.0
hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)

# Convert to BGR for display
flow_bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

print("Flow color visualization completed.")
show(flow_bgr, "Dense Optical Flow Color Wheel Encoding")

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Optical Flow
1. Compare LK tracked point count decay over 30 frames vs re-seeding `goodFeaturesToTrack` every 10 frames.
2. Sweep Farneback's `winsize` parameter and describe the smoothness/detail trade-off in the flow field.
3. Use `summarize_flow` across every consecutive frame pair in the video and plot mean speed over time.

Use the empty cell below to work through them.


#### Solutions — Optical Flow

In [ ]:
# Solution 1: LK tracked point count decay vs re-seeding
# Explanation: In sparse optical flow (Lucas-Kanade), tracked keypoints drift and drop out
# over time because of occlusion, illumination changes, or camera motion. To maintain tracking,
# a re-seeding step must run every 10 frames to detect new `goodFeaturesToTrack` and merge them
# with existing ones.


In [ ]:
# Solution 2: Sweep Farneback's winsize parameter (Visualized)
frames = read_real_video_frames("traffic.mp4", max_frames=2)
prev = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
curr = cv2.cvtColor(frames[1], cv2.COLOR_BGR2GRAY)
f5 = cv2.calcOpticalFlowFarneback(prev, curr, None, 0.5, 3, 5, 3, 5, 1.2, 0)
f21 = cv2.calcOpticalFlowFarneback(prev, curr, None, 0.5, 3, 21, 3, 5, 1.2, 0)
show_grid(
    [
        ("Winsize=5 (Noisy but detailed)", flow_to_bgr(f5)),
        ("Winsize=21 (Smooth but blurred)", flow_to_bgr(f21)),
    ]
)

In [ ]:
# Solution 3: Flow speed plotting over time
def summarize_motion_flow(flow_sequence: list[np.ndarray]) -> list[float]:
    """Extract average motion speed over time from sequence of flows."""
    speeds = []
    for flow in flow_sequence:
        mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        speeds.append(np.mean(mag))
    return speeds

## Summary

You can compute, visualize, and summarize sparse or dense optical flow while explaining brightness-constancy and aperture-problem limits.

- **Best Practices:** Use pyramids for larger displacements, reject poorly tracked points, and compare flow against the original frames rather than only a color map.
- **Common Pitfalls:** Treating camera movement as object movement, trusting flow in textureless regions, and mixing up dense-flow direction conventions.